# Extended Data Cleaning Pipeline — 2023 Q1 to 2025 Q4
**Purpose:** Re-run the same cleaning logic from `unit2-cleaning.ipynb` using the extended
raw files that cover 2023–2025 (3 full years, 12 quarters).  
**Output:** Three cleaned CSV files in `datasets/extended/` ready for feature engineering.

## 0. Setup

In [ ]:
import pandas as pd
import os

# Extended raw files (downloaded by fetch script — 2023-01 to 2025-12)
GDP_PATH = 'datasets/OECD.SDD.NAD,DSD_NAMAIN1@DF_QNA_EXPENDITURE_CAPITA,+Q_2023_2025.csv'
UNE_PATH = 'datasets/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,+..._Z.Y._T.Y_GE15..M_2023_2025.csv'

AGGREGATE_AREAS = {'EA20', 'EU27_2020', 'G7', 'OECD', 'OECDE', 'USMCA'}

os.makedirs('datasets/extended', exist_ok=True)

gdp_raw = pd.read_csv(GDP_PATH, low_memory=False)
une_raw = pd.read_csv(UNE_PATH, low_memory=False)

print('GDP raw shape :', gdp_raw.shape)
print('UNE raw shape :', une_raw.shape)

## 1. Duplicate Detection

In [ ]:
print('Exact duplicates — GDP:', gdp_raw.duplicated().sum(), '| UNE:', une_raw.duplicated().sum())

gdp_key_dup = gdp_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD']).sum()
une_key_dup = une_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD']).sum()
print('Key-level duplicates (REF_AREA + TIME_PERIOD) — GDP:', gdp_key_dup, '| UNE:', une_key_dup)

# Show the PRICE_BASE structural duplicate pattern
mask = gdp_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD'], keep=False)
print('\nSample: two PRICE_BASE variants per country-period:')
print(gdp_raw[mask][['REF_AREA', 'TIME_PERIOD', 'PRICE_BASE', 'OBS_VALUE']].head(6).to_string())
print('\n-> Treatment: keep LR (constant 2020 prices) only.')

## 2. Column Selection

In [ ]:
GDP_KEEP = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'PRICE_BASE', 'OBS_STATUS', 'REF_YEAR_PRICE']
UNE_KEEP = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS']

gdp_clean = gdp_raw[GDP_KEEP].copy()
une_clean = une_raw[UNE_KEEP].copy()

print('GDP after column selection:', gdp_clean.shape)
print('UNE after column selection:', une_clean.shape)
print('\nMissing values — GDP:\n', gdp_clean.isnull().sum())
print('\nMissing values — UNE:\n', une_clean.isnull().sum())

## 3. Filtering and Time Alignment
### 3.1 Keep constant-price series and remove aggregate areas

In [ ]:
print('PRICE_BASE distribution:', gdp_clean['PRICE_BASE'].value_counts().to_dict())
gdp_clean = gdp_clean[gdp_clean['PRICE_BASE'] == 'LR'].drop(columns=['PRICE_BASE'])
print('After PRICE_BASE=LR filter:', gdp_clean.shape)

agg_found = sorted(gdp_clean[gdp_clean['REF_AREA'].isin(AGGREGATE_AREAS)]['REF_AREA'].unique())
print('Aggregate areas removed:', agg_found)
gdp_clean = gdp_clean[~gdp_clean['REF_AREA'].isin(AGGREGATE_AREAS)]
print('After removing aggregates:', gdp_clean.shape)

print('\nOBS_STATUS distribution (data quality):')
print(gdp_clean['OBS_STATUS'].value_counts())
print('\nNote: E=estimated, P=provisional, A=final. 2023 data may have more E/P entries.')

### 3.2 Parse and align time periods

In [ ]:
# GDP: YYYY-QN -> YYYYQN period string
gdp_clean['QUARTER'] = pd.PeriodIndex(
    gdp_clean['TIME_PERIOD'].str.replace('-Q', 'Q'), freq='Q'
).astype(str)
gdp_clean = gdp_clean.rename(columns={'OBS_VALUE': 'GDP_PER_CAPITA_USD_PPP'})

# UNE: monthly -> quarterly mean
une_clean['QUARTER'] = pd.to_datetime(une_clean['TIME_PERIOD'], format='%Y-%m').dt.to_period('Q').astype(str)
une_quarterly = (
    une_clean
    .groupby(['REF_AREA', 'QUARTER'], as_index=False)['OBS_VALUE']
    .mean()
    .rename(columns={'OBS_VALUE': 'UNE_RATE_PCT'})
)

print('Quarters in GDP:', sorted(gdp_clean['QUARTER'].unique()))
print('Quarters in UNE:', sorted(une_quarterly['QUARTER'].unique()))

## 4. Merge and Export

In [ ]:
gdp_export = gdp_clean[['REF_AREA', 'QUARTER', 'GDP_PER_CAPITA_USD_PPP', 'OBS_STATUS', 'REF_YEAR_PRICE']].copy()
une_export = une_quarterly.copy()

merged = pd.merge(gdp_export, une_export, on=['REF_AREA', 'QUARTER'], how='inner')

print('GDP cleaned    :', gdp_export.shape, ' countries:', gdp_export['REF_AREA'].nunique())
print('UNE cleaned    :', une_export.shape, ' countries:', une_export['REF_AREA'].nunique())
print('Merged (final) :', merged.shape, ' countries:', merged['REF_AREA'].nunique(), ' quarters:', merged['QUARTER'].nunique())
print('Quarters       :', sorted(merged['QUARTER'].unique()))
print('Missing values :', merged.isnull().sum().to_dict())

gdp_export.to_csv('datasets/extended/gdp_per_capita_cleaned_extended.csv', index=False, sep=';')
une_export.to_csv('datasets/extended/unemployment_quarterly_cleaned_extended.csv', index=False, sep=';')
merged.to_csv('datasets/extended/gdp_unemployment_merged_extended.csv', index=False, sep=';')

print('\nFiles saved to datasets/extended/')
print(merged.sort_values(['REF_AREA', 'QUARTER']).head(16).to_string())